# Uh ROI 引导的 Udeco 边沿实验

本实验严格配对 Uh2→Udeco2、Uh3→Udeco3。Uh 只确定事件 ROI；Udeco 在完整 ROI 内独立检测候选，再回到原信号定位、计算形态并各选最大幅度 rise/fall。图表均可缩放、平移。每个阶段代码 Cell 顶部都有 local override；改动后只需重跑该 Cell 及后续 Cell。

In [ ]:
# Cell 2：全局默认值、依赖和通道语义
from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import clear_output, display

from udeco_uh_roi_edge_detection import (
    ExperimentConfig, baseline_sigmas, build_direct_event_table,
    detect_roi_candidates, estimate_motion_frequency, find_uh_rois, load_paired_sample,
    localize_roi_candidates, preprocess_channel, recover_missing_extrema_pairs, resolve_motion_interval,
)

pio.renderers.default = "plotly_mimetype"
SAMPLE_NAME = "cgx_rope_1"
DATA_ROOT = Path("subject_data")
FS = 100
BASELINE_SEC = 15.0
MEDIAN_KERNEL = 4
CONTEXT_WINDOW_SEC = 10.0
CONTEXT_ROBUST_ITERS = 1

UH_PEAK_PROMINENCE_SIGMA = 4.0
UH_PEAK_TO_COARSE_MEAN_SIGMA = 20.0  # Nbar：peak - (coarse L + coarse R)/2 必须严格大于 Nbar × 前15 s sigma_uh
UH_PEAK_PERIOD_FRACTION = 0.75  # 主峰最小间隔 = ceil(percentF * fs / f₀) 个采样点
UH_FLAT_SLOPE_SIGMA = 2.0
UH_FLAT_MIN_SEC = 0.05
FFT_MIN_HZ = 0.5
FFT_MAX_HZ = 5.0
MOTION_START_SEC = None
MOTION_END_SEC = None
F0_OVERRIDE_HZ = {2: None, 3: None}
UDECO_LEFT_LEAD_SEC = 0.03
ROI_FILL_COLORS = ("#91d5c4", "#a9bff0")  # 相邻逻辑 ROI 交替底色
ROI_LEFT_LEAD_COLOR = "#f4b26e"  # 仅 Udeco 左拓范围

CANDIDATE_MERGE_SEC = 0.02  # 跨 ROI 重复候选仅作诊断

LOCALIZE_SEARCH_SEC = 0.20  # 极值附近选择导数主峰
EXTREMA_TRACE_SEC = 1.0  # 极值边沿追踪窗口，可跨越 ROI 与左拓边界
THRESHOLD_DU = 2.0  # 弱导数峰保护和单调性容差，使用前15 s sigma_du
THRESHOLD_DU_SHARP_LEFT = 2.0
THRESHOLD_DUPEAK_SHARP_LEFT = 0.15
THRESHOLD_DU_SHARP_RIGHT = 3.0
THRESHOLD_DUPEAK_SHARP_RIGHT = 0.40
THRESHOLD_DU_GENERAL_LEFT = 2.0
THRESHOLD_DUPEAK_GENERAL_LEFT = 0.08
THRESHOLD_DU_GENERAL_RIGHT = 2.0
THRESHOLD_DUPEAK_GENERAL_RIGHT = 0.08
SHARP_NEIGHBOR_RATIO = 0.5  # 两邻点并非都大于正峰值的一半

MIN_AMPLITUDE_SIGMA_K = 2.0
DELTA_MISMATCH_FRACTION = 0.5
DERIV_PEAK_WIDTH_THRESHOLD = 0.04
DERIV_PEAK_SIGMA_K = 5.0
DIRECTION_MAP = {
    2: {1: {"sharp": "detach", "gradual": "release"},
        -1: {"sharp": "attach", "gradual": "force"}},
    3: {1: {"sharp": "detach", "gradual": "release"},
        -1: {"sharp": "attach", "gradual": "force"}},
}
PLOT_START_SEC = None
PLOT_END_SEC = None
BASE_CONFIG = ExperimentConfig(
    median_kernel=MEDIAN_KERNEL, context_window_sec=CONTEXT_WINDOW_SEC,
    context_robust_iters=CONTEXT_ROBUST_ITERS, baseline_sec=BASELINE_SEC,
    uh_peak_prominence_sigma=UH_PEAK_PROMINENCE_SIGMA,
    uh_peak_to_coarse_mean_sigma=UH_PEAK_TO_COARSE_MEAN_SIGMA,
    uh_peak_period_fraction=UH_PEAK_PERIOD_FRACTION,
    uh_flat_slope_sigma=UH_FLAT_SLOPE_SIGMA, uh_flat_min_sec=UH_FLAT_MIN_SEC,
    fft_min_hz=FFT_MIN_HZ, fft_max_hz=FFT_MAX_HZ,
    udeco_left_lead_sec=UDECO_LEFT_LEAD_SEC,
    candidate_merge_sec=CANDIDATE_MERGE_SEC,
    localize_search_sec=LOCALIZE_SEARCH_SEC, extrema_trace_sec=EXTREMA_TRACE_SEC,
    threshold_du=THRESHOLD_DU,
    threshold_du_sharp_left=THRESHOLD_DU_SHARP_LEFT,
    threshold_dupeak_sharp_left=THRESHOLD_DUPEAK_SHARP_LEFT,
    threshold_du_sharp_right=THRESHOLD_DU_SHARP_RIGHT,
    threshold_dupeak_sharp_right=THRESHOLD_DUPEAK_SHARP_RIGHT,
    threshold_du_general_left=THRESHOLD_DU_GENERAL_LEFT,
    threshold_dupeak_general_left=THRESHOLD_DUPEAK_GENERAL_LEFT,
    threshold_du_general_right=THRESHOLD_DU_GENERAL_RIGHT,
    threshold_dupeak_general_right=THRESHOLD_DUPEAK_GENERAL_RIGHT,
    sharp_neighbor_ratio=SHARP_NEIGHBOR_RATIO,
    min_amplitude_sigma_k=MIN_AMPLITUDE_SIGMA_K,
    delta_mismatch_fraction=DELTA_MISMATCH_FRACTION,
    deriv_peak_width_threshold_sec=DERIV_PEAK_WIDTH_THRESHOLD,
    deriv_peak_sigma_k=DERIV_PEAK_SIGMA_K,
)
print("Configured", SAMPLE_NAME, "at", FS, "Hz; detector: derivative_extrema")

## 1. 数据读取与四路 raw signals

使用现有 Pydisplay 序号修复和 Udeco 标定。以下四图共用时间 x 轴，灰色区域为前 15 s 静止佩戴基线。

In [ ]:
# Local override：数据与初始显示范围
sample_name = SAMPLE_NAME
fs = FS
data_root = DATA_ROOT
plot_start_sec = PLOT_START_SEC
plot_end_sec = PLOT_END_SEC

matches = sorted(data_root.glob(f"*/{sample_name}_sensor.csv"))
if len(matches) != 1:
    raise FileNotFoundError(f"Expected one sensor CSV for {sample_name}, found {matches}")
sample = load_paired_sample(matches[0], data_root=data_root, fs=fs)
time_s = sample["time_s"]
fs = sample["fs"]
raw_signals = sample["signals"]
print(sample["sample_name"], sample["subject"], len(time_s), "samples")
raw_fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.045,
                        subplot_titles=list(raw_signals))
for row, (name, values) in enumerate(raw_signals.items(), start=1):
    raw_fig.add_trace(go.Scattergl(x=time_s, y=values, name=name, line=dict(width=1)), row=row, col=1)
    raw_fig.add_vrect(x0=0, x1=BASELINE_SEC, fillcolor="silver", opacity=0.15,
                      line_width=0, row=row, col=1)
    raw_fig.update_yaxes(title_text=name, row=row, col=1)
raw_fig.update_xaxes(title_text="Time (s)", row=4, col=1)
if plot_start_sec is not None and plot_end_sec is not None:
    raw_fig.update_xaxes(range=[plot_start_sec, plot_end_sec])
raw_fig.update_layout(height=1050, title=f"{sample_name}: four raw channels",
                      template="plotly_white", dragmode="pan", hovermode="x unified")
raw_fig.show(config={"scrollZoom": True, "displaylogo": False})

## 2. 4 点 median、Bcontext 与固定基线

四路都按 raw → clean → Bcontext → clean−Bcontext 处理。后续所有噪声阈值由前 15 s 给出，绝不在事件附近重估计。

In [ ]:
# Local override：预处理和基线
median_kernel = MEDIAN_KERNEL
context_window_sec = CONTEXT_WINDOW_SEC
context_robust_iters = CONTEXT_ROBUST_ITERS
baseline_sec = BASELINE_SEC

preprocess_config = replace(BASE_CONFIG, median_kernel=median_kernel,
                            context_window_sec=context_window_sec,
                            context_robust_iters=context_robust_iters,
                            baseline_sec=baseline_sec)
processed = {name: preprocess_channel(values, fs, preprocess_config)
             for name, values in raw_signals.items()}
noise = {channel: baseline_sigmas(processed[f"Uh{channel}"]["edge_signal"],
                                  processed[f"Udeco{channel}"]["edge_signal"],
                                  fs=fs, baseline_sec=baseline_sec)
         for channel in (2, 3)}
for channel, n in noise.items():
    print(f"Pair {channel}: sigma_uh={n['sigma_uh']:.6g}, sigma_udeco={n['sigma_udeco']:.6g}, "
          f"sigma_du={n['sigma_du']:.6g}, sigma_duh={n['sigma_duh']:.6g} (first {baseline_sec:g} s)")

def plot_preprocessed(name):
    values = processed[name]
    keys = ("raw", "clean", "Bcontext", "edge_signal")
    fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.04,
                        subplot_titles=("raw", "4-point median clean", "Bcontext", "clean - Bcontext"))
    for row, key in enumerate(keys, start=1):
        fig.add_trace(go.Scattergl(x=time_s, y=values[key], name=key,
                                   line=dict(width=1)), row=row, col=1)
        fig.add_vrect(x0=0, x1=baseline_sec, fillcolor="silver", opacity=0.15,
                      line_width=0, row=row, col=1)
    fig.update_xaxes(title_text="Time (s)", row=4, col=1)
    fig.update_layout(height=900, title=f"{sample_name}: {name} preprocessing",
                      template="plotly_white", dragmode="pan", hovermode="x unified")
    fig.show(config={"scrollZoom": True, "displaylogo": False})
    return fig

preprocessing_figures = {name: plot_preprocessed(name) for name in raw_signals}

## 3. ACC 运动段、Uh FFT、主峰选择与逻辑 ROI

仅在运动段内寻找 Uh 正峰。`find_peaks(distance=ceil(percentF × fs / f₀))` 直接选择主峰，再要求 `peak − (coarse L + coarse R)/2 > Nbar × 前15 s sigma_uh`；无最小间隔的候选峰只用于诊断。每个主峰单独建 ROI：短于等于 `1.5/f₀` 的相邻周期按峰间谷底覆盖，长间隔保留 refineL/refineR 并留空。频谱最大峰只是 f₀ 候选，可手动覆盖。

In [ ]:
# Local override：运动段、FFT、Uh 正峰及 ROI
motion_start_sec = MOTION_START_SEC
motion_end_sec = MOTION_END_SEC
f0_override_hz = F0_OVERRIDE_HZ.copy()
fft_min_hz = FFT_MIN_HZ
fft_max_hz = FFT_MAX_HZ
peak_prominence_sigma = UH_PEAK_PROMINENCE_SIGMA
peak_to_coarse_mean_sigma = UH_PEAK_TO_COARSE_MEAN_SIGMA  # Nbar
peak_period_fraction = UH_PEAK_PERIOD_FRACTION
flat_slope_sigma = UH_FLAT_SLOPE_SIGMA
flat_min_sec = UH_FLAT_MIN_SEC
udeco_left_lead_sec = UDECO_LEFT_LEAD_SEC
roi_fill_colors = ROI_FILL_COLORS

roi_config = replace(preprocess_config,
                     uh_peak_prominence_sigma=peak_prominence_sigma,
                     uh_peak_to_coarse_mean_sigma=peak_to_coarse_mean_sigma,
                     uh_peak_period_fraction=peak_period_fraction,
                     uh_flat_slope_sigma=flat_slope_sigma,
                     uh_flat_min_sec=flat_min_sec,
                     fft_min_hz=fft_min_hz, fft_max_hz=fft_max_hz,
                     udeco_left_lead_sec=udeco_left_lead_sec)
motion = resolve_motion_interval(sample, baseline_sec=baseline_sec,
                                 manual_start_sec=motion_start_sec,
                                 manual_end_sec=motion_end_sec)
print(f"Motion: {motion['start_sec']:.2f}–{motion['end_sec']:.2f} s ({motion['source']})")
roi_results = {}
for channel in (2, 3):
    edge = processed[f"Uh{channel}"]["edge_signal"]
    spectrum = estimate_motion_frequency(edge, fs=fs, motion_interval=motion,
                                         min_hz=fft_min_hz, max_hz=fft_max_hz,
                                         override_hz=f0_override_hz[channel])
    rois, peaks = find_uh_rois(edge, fs=fs, sigma_uh=noise[channel]["sigma_uh"],
                               sigma_duh=noise[channel]["sigma_duh"],
                               motion_interval=motion, f0_hz=spectrum["f0_hz"],
                               config=roi_config)
    roi_results[channel] = {"rois": rois, "peaks": peaks, "spectrum": spectrum}
    min_distance_points = max(1, int(np.ceil(peak_period_fraction * fs / spectrum["f0_hz"])))
    print(f"Uh{channel}: f0={spectrum['f0_hz']:.3f} Hz ({spectrum['source']}), "
          f"distance={min_distance_points} samples; {len(peaks)} candidates → {len(rois)} main peaks/ROIs")
    fft_fig = go.Figure()
    band = (spectrum["frequencies_hz"] >= fft_min_hz) & (spectrum["frequencies_hz"] <= fft_max_hz)
    fft_fig.add_trace(go.Scatter(x=spectrum["frequencies_hz"][band],
                                 y=spectrum["power"][band], name="Uh FFT power"))
    fft_fig.add_vline(x=spectrum["f0_hz"], line_color="#e41a1c", line_dash="dash",
                      annotation_text=f"f₀={spectrum['f0_hz']:.3f} Hz")
    fft_fig.update_layout(title=f"Uh{channel}: motion spectrum", xaxis_title="Frequency (Hz)",
                          yaxis_title="Power", template="plotly_white", height=350, dragmode="pan")
    fft_fig.show(config={"scrollZoom": True, "displaylogo": False})
    fig = go.Figure()
    fig.add_trace(go.Scattergl(x=time_s, y=edge, name=f"Uh{channel} edge_signal",
                               line=dict(color="#444", width=1)))
    main_peaks = [item for item in peaks if item["is_main"]]
    minor_peaks = [item for item in peaks if not item["is_main"]]
    minor_indices = [item["peak_idx"] for item in minor_peaks]
    fig.add_trace(go.Scattergl(x=time_s[minor_indices], y=edge[minor_indices], mode="markers",
                               name="candidate small peaks",
                               text=[item["roi_id"] or "unassigned" for item in minor_peaks],
                               marker=dict(color="#999999", size=5, opacity=0.6),
                               hovertemplate="t=%{x:.3f}s; ROI=%{text}<extra></extra>"))
    peak_indices = [item["peak_idx"] for item in main_peaks]
    threshold_y = [(edge[item["coarse_left_idx"]] + edge[item["coarse_right_idx"]]) / 2 + peak_to_coarse_mean_sigma * noise[channel]["sigma_uh"] for item in main_peaks]
    fig.add_trace(go.Scattergl(x=time_s[peak_indices], y=threshold_y, mode="markers",
                               name=f"mean(coarse L/R) + {peak_to_coarse_mean_sigma:g}σ_uh",
                               marker=dict(symbol="cross", color="#e41a1c", size=9)))
    fig.add_trace(go.Scattergl(x=time_s[peak_indices], y=edge[peak_indices], mode="markers",
                               name="main peaks", text=[item["roi_id"] for item in main_peaks],
                               marker=dict(symbol="diamond", color="#e41a1c", size=10),
                               hovertemplate="t=%{x:.3f}s; ROI=%{text}<extra></extra>"))
    for field, label, symbol, color in (
        ("coarse_left_idx", "coarse L", "circle-open", "#ff7f00"),
        ("coarse_right_idx", "coarse R", "circle-open", "#ff7f00"),
        ("refined_left_idx", "refined L", "triangle-up", "#377eb8"),
        ("refined_right_idx", "refined R", "triangle-down", "#4daf4a"),
    ):
        indices = [item[field] for item in main_peaks]
        fig.add_trace(go.Scattergl(x=time_s[indices], y=edge[indices], mode="markers",
                                   name=label, marker=dict(symbol=symbol, color=color, size=8),
                                   customdata=[[item["prominence"], item["coarse_mean_rise"]] for item in main_peaks],
                                   hovertemplate="t=%{x:.3f}s; peak - mean(coarse L/R)=%{customdata[1]:.5g}; prominence=%{customdata[0]:.5g}<extra></extra>"))
    shapes = [dict(type="rect", xref="x", yref="paper",
                   x0=time_s[roi["left_idx"]], x1=time_s[roi["right_idx"]],
                   y0=0, y1=1, fillcolor=roi_fill_colors[i % 2], opacity=0.22, line_width=0)
              for i, roi in enumerate(rois)]
    shapes.extend(dict(type="line", xref="x", yref="paper",
                       x0=time_s[roi["split_right_idx"]], x1=time_s[roi["split_right_idx"]],
                       y0=0, y1=1, line=dict(color="#984ea3", dash="dot"))
                  for roi in rois if roi["split_right_idx"] is not None)
    fig.update_layout(shapes=shapes)
    fig.update_layout(title=f"Uh{channel}: peaks → refined bounds → disjoint ROIs",
                      xaxis_title="Time (s)", yaxis_title="clean - Bcontext",
                      height=550, template="plotly_white", dragmode="pan")
    fig.show(config={"scrollZoom": True, "displaylogo": False})

## 4. Uh/Udeco 配对与 ROI 范围检查

每一对共享时间轴：青色/蓝色交替表示相邻的 Uh 逻辑 ROI，橙色仅表示 Udeco 左拓搜索区。灰点是通过基线阈值但未选为主峰的候选峰，红点是每个 ROI 的主峰。

In [ ]:
# Local override：Uh/Udeco 配对图显示范围
pair_plot_start_sec = PLOT_START_SEC if PLOT_START_SEC is not None else motion["start_sec"]
pair_plot_end_sec = PLOT_END_SEC if PLOT_END_SEC is not None else motion["end_sec"]
roi_fill_colors = ROI_FILL_COLORS
left_lead_color = ROI_LEFT_LEAD_COLOR
plot_lo = max(0, round(pair_plot_start_sec * fs))
plot_hi = min(len(time_s), round(pair_plot_end_sec * fs) + 1)
plot_indices = np.arange(plot_lo, plot_hi)

for channel in (2, 3):
    rois = roi_results[channel]["rois"]
    peaks = roi_results[channel]["peaks"]
    uh = processed[f"Uh{channel}"]["edge_signal"]
    udeco = processed[f"Udeco{channel}"]["edge_signal"]
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.07,
                        subplot_titles=(f"Uh{channel} edge_signal / ROI",
                                        f"Udeco{channel} edge_signal / search"))
    fig.add_trace(go.Scattergl(x=time_s[plot_indices], y=uh[plot_indices], name=f"Uh{channel}",
                               line=dict(width=1)), row=1, col=1)
    fig.add_trace(go.Scattergl(x=time_s[plot_indices], y=udeco[plot_indices], name=f"Udeco{channel}",
                               line=dict(width=1)), row=2, col=1)
    visible_rois = [(i, roi) for i, roi in enumerate(rois)
                    if roi["right_idx"] >= plot_lo and roi["search_left_idx"] < plot_hi]
    shapes = []
    for roi_index, roi in visible_rois:
        left, right = time_s[roi["left_idx"]], time_s[roi["right_idx"]]
        for xref, yref in (("x", "y domain"), ("x2", "y2 domain")):
            shapes.append(dict(type="rect", xref=xref, yref=yref, x0=left, x1=right,
                               y0=0, y1=1, fillcolor=roi_fill_colors[roi_index % 2], opacity=0.22,
                               line_width=0))
        if roi["search_left_idx"] < roi["left_idx"]:
            shapes.append(dict(type="rect", xref="x2", yref="y2 domain",
                               x0=time_s[roi["search_left_idx"]], x1=left,
                               y0=0, y1=1, fillcolor=left_lead_color, opacity=0.35,
                               line_width=0))
    fig.update_layout(shapes=shapes)
    minor_peaks = [item for item in peaks if not item["is_main"]
                   and plot_lo <= item["peak_idx"] < plot_hi]
    minor_indices = [item["peak_idx"] for item in minor_peaks]
    fig.add_trace(go.Scattergl(x=time_s[minor_indices], y=uh[minor_indices], mode="markers",
                               name="Uh candidate small peaks",
                               text=[item["roi_id"] or "unassigned" for item in minor_peaks],
                               marker=dict(color="#999999", size=5, opacity=0.6),
                               hovertemplate="t=%{x:.3f}s; ROI=%{text}<extra></extra>"), row=1, col=1)
    peak_indices = [roi["main_peak_idx"] for _, roi in visible_rois]
    fig.add_trace(go.Scattergl(x=time_s[peak_indices], y=uh[peak_indices], mode="markers",
                               name="Uh main peaks", text=[roi["roi_id"] for _, roi in visible_rois],
                               marker=dict(color="#e41a1c", size=7)), row=1, col=1)
    fig.update_xaxes(range=[pair_plot_start_sec, pair_plot_end_sec])
    fig.update_xaxes(title_text="Time (s)", row=2, col=1)
    fig.update_layout(title=f"Uh{channel} → Udeco{channel}: logical ROI and left lead",
                      template="plotly_white", height=700, dragmode="pan",
                      hovermode="x unified")
    fig.show(config={"scrollZoom": True, "displaylogo": False})

## 5. ROI 内差分极值候选发现

默认在每个 Udeco 搜索范围（含橙色左拓）内分别取相邻采样差的最大正值和最小负值，每个方向至多一个候选。中图是原始差分，下图是以**前 15 s** `sigma_du` 归一化的差分；红/蓝点是检出的上升/下降候选。虚线 `±THRESHOLD_DU` 仅供观察定位阶段的导数强度，**极值检测本身不使用阈值筛选**。


In [ ]:
# Local override：极值候选发现
roi_fill_colors = ROI_FILL_COLORS
left_lead_color = ROI_LEFT_LEAD_COLOR
deriv_reference_sigma_k = THRESHOLD_DU
candidate_merge_sec = CANDIDATE_MERGE_SEC

detect_config = replace(roi_config, candidate_merge_sec=candidate_merge_sec)
detections = {}
for channel in (2, 3):
    rois = roi_results[channel]["rois"]
    edge = processed[f"Udeco{channel}"]["edge_signal"]
    candidates, diagnostics = detect_roi_candidates(edge, rois, fs=fs, config=detect_config)
    detections[channel] = {"candidates": candidates, "diagnostics": diagnostics}
    directions_by_roi = {roi["roi_id"]: set() for roi in rois}
    for candidate in candidates:
        directions_by_roi[candidate["roi_id"]].add(candidate["direction"])
    both_count = sum(directions == {1, -1} for directions in directions_by_roi.values())
    print(f"Udeco{channel}: {len(candidates)} derivative_extrema candidates; "
          f"{both_count}/{len(rois)} ROIs have both directions")
    derivative = diagnostics["derivative_signal"]
    score = diagnostics["derivative_score"]
    sigma_du = diagnostics["derivative_sigma"]
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.055,
                        subplot_titles=("Udeco edge_signal", "dU = sample difference",
                                        "dU / baseline sigma_du"))
    fig.add_trace(go.Scattergl(x=time_s, y=edge, name="Udeco edge_signal",
                               line=dict(color="#444", width=1)), row=1, col=1)
    fig.add_trace(go.Scattergl(x=time_s, y=derivative, name="dU",
                               line=dict(color="#666", width=1)), row=2, col=1)
    fig.add_trace(go.Scattergl(x=time_s, y=score, name="dU / sigma_du",
                               line=dict(color="#1f78b4", width=1)), row=3, col=1)
    for sign in (-1, 1):
        fig.add_hline(y=sign * deriv_reference_sigma_k * sigma_du,
                      line_dash="dot", line_color="#888", row=2, col=1)
        fig.add_hline(y=sign * deriv_reference_sigma_k,
                      line_dash="dash", line_color="red", row=3, col=1)
    for direction, label, color, symbol in ((1, "rising candidates", "#e41a1c", "triangle-up"),
                                            (-1, "falling candidates", "#377eb8", "triangle-down")):
        chosen = [candidate for candidate in candidates if candidate["direction"] == direction]
        centers = [candidate["center_idx"] for candidate in chosen]
        fig.add_trace(go.Scattergl(x=time_s[centers], y=edge[centers], mode="markers",
                                   name=f"{label} on Udeco", text=[c["roi_id"] for c in chosen],
                                   marker=dict(color=color, symbol=symbol, size=9),
                                   showlegend=False), row=1, col=1)
        fig.add_trace(go.Scattergl(x=time_s[centers], y=derivative[centers], mode="markers",
                                   name=f"{label} on dU", marker=dict(color=color, symbol=symbol, size=9),
                                   showlegend=False), row=2, col=1)
        fig.add_trace(go.Scattergl(x=time_s[centers], y=score[centers], mode="markers",
                                   name=label, text=[c["roi_id"] for c in chosen],
                                   marker=dict(color=color, symbol=symbol, size=10),
                                   hovertemplate="%{text}: t=%{x:.3f}s, score=%{y:.1f}<extra></extra>"),
                      row=3, col=1)
    shapes = list(fig.layout.shapes)
    for roi_index, roi in enumerate(rois):
        shapes.append(dict(type="rect", xref="x", yref="y domain",
                           x0=time_s[roi["left_idx"]], x1=time_s[roi["right_idx"]],
                           y0=0, y1=1, fillcolor=roi_fill_colors[roi_index % 2],
                           opacity=0.18, line_width=0))
        if roi["search_left_idx"] < roi["left_idx"]:
            shapes.append(dict(type="rect", xref="x", yref="y domain",
                               x0=time_s[roi["search_left_idx"]], x1=time_s[roi["left_idx"]],
                               y0=0, y1=1, fillcolor=left_lead_color,
                               opacity=0.32, line_width=0))
    fig.update_layout(shapes=shapes, title=f"Udeco{channel}: derivative extrema candidates",
                      height=950, template="plotly_white", dragmode="pan", hovermode="x unified")
    fig.update_xaxes(title_text="Time (s)", row=3, col=1)
    fig.show(config={"scrollZoom": True, "displaylogo": False})


## 6. 每个 candidate 的局部定位与缺失边沿回退

正峰两邻点并非都大于半峰值时走尖锐规则：左侧首次非正差分或连续 2 个低差分，右侧连续 2 个低差分。其余正峰和负峰走一般规则：左侧第 2 个反向差分或连续 3 个低差分，右侧第 2 个反向差分或连续 4 个低差分。若 d[k]=U[k+1]−U[k]，尖锐边沿连续低差分 d[a:a+2] 的起点为 U[a+2]、终点为 U[a]；一般边沿左侧连续 3 点的起点为 U[a+2]、右侧连续 4 点的终点为 U[a+3]。各方向阈值使用前 15 s 的 sigma_du；定位可跨 ROI，最多追踪 1 s。重叠边沿共享最小 |dU| 分割采样，零长度时按第 2 个反向差分重新追踪。缺少已定位的一升一降时才尝试峰值回退。


In [ ]:
# Local override：局部定位与完整边沿
localize_search_sec = LOCALIZE_SEARCH_SEC
extrema_trace_sec = EXTREMA_TRACE_SEC
threshold_du = THRESHOLD_DU
threshold_du_sharp_left = THRESHOLD_DU_SHARP_LEFT
threshold_dupeak_sharp_left = THRESHOLD_DUPEAK_SHARP_LEFT
threshold_du_sharp_right = THRESHOLD_DU_SHARP_RIGHT
threshold_dupeak_sharp_right = THRESHOLD_DUPEAK_SHARP_RIGHT
threshold_du_general_left = THRESHOLD_DU_GENERAL_LEFT
threshold_dupeak_general_left = THRESHOLD_DUPEAK_GENERAL_LEFT
threshold_du_general_right = THRESHOLD_DU_GENERAL_RIGHT
threshold_dupeak_general_right = THRESHOLD_DUPEAK_GENERAL_RIGHT
sharp_neighbor_ratio = SHARP_NEIGHBOR_RATIO

localize_config = replace(detect_config, localize_search_sec=localize_search_sec,
                          extrema_trace_sec=extrema_trace_sec,
                          threshold_du=threshold_du,
                          threshold_du_sharp_left=threshold_du_sharp_left,
                          threshold_dupeak_sharp_left=threshold_dupeak_sharp_left,
                          threshold_du_sharp_right=threshold_du_sharp_right,
                          threshold_dupeak_sharp_right=threshold_dupeak_sharp_right,
                          threshold_du_general_left=threshold_du_general_left,
                          threshold_dupeak_general_left=threshold_dupeak_general_left,
                          threshold_du_general_right=threshold_du_general_right,
                          threshold_dupeak_general_right=threshold_dupeak_general_right,
                          sharp_neighbor_ratio=sharp_neighbor_ratio)
localized = {}
for channel in (2, 3):
    signal = processed[f"Udeco{channel}"]
    localized[channel] = localize_roi_candidates(
        signal["clean"], signal["edge_signal"], detections[channel]["candidates"],
        roi_results[channel]["rois"], fs=fs, noise=noise[channel], config=localize_config)
    localized[channel] = recover_missing_extrema_pairs(
        signal["edge_signal"], localized[channel], roi_results[channel]["rois"],
        fs=fs, noise=noise[channel], config=localize_config)
    fallback_count = sum(c["detector_name"] == "peak_fallback" for c in localized[channel])
    print(f"Udeco{channel}: {sum(c['resolved'] for c in localized[channel])}/{len(localized[channel])} candidates localized; {fallback_count} fallback edges")

def plot_candidate_local(channel, candidate_index):
    item = localized[channel][candidate_index]
    signal = processed[f"Udeco{channel}"]
    roi = next(r for r in roi_results[channel]["rois"] if r["roi_id"] == item["roi_id"])
    center = item["center_idx"]
    markers = [item.get(k) for k in ("start_idx", "end_idx",
                                    "derivative_fwhm_left_idx", "derivative_fwhm_right_idx")]
    valid = [int(m) for m in markers if m is not None and pd.notna(m)]
    lo = max(0, min([center - round(0.25 * fs), *valid]))
    hi = min(len(time_s) - 1, max([center + round(0.35 * fs), *valid]))
    indices = np.arange(lo, hi + 1)
    derivative = np.diff(signal["edge_signal"], prepend=signal["edge_signal"][0])
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10,
                        specs=[[{"secondary_y": True}], [{}]],
                        subplot_titles=("Udeco clean / edge_signal", "diff(edge_signal) morphology"))
    fig.add_trace(go.Scatter(x=time_s[indices], y=signal["clean"][indices], name="clean"),
                  row=1, col=1, secondary_y=False)
    fig.add_trace(go.Scatter(x=time_s[indices], y=signal["edge_signal"][indices],
                             name="edge_signal"), row=1, col=1, secondary_y=True)
    fig.add_trace(go.Scatter(x=time_s[indices], y=derivative[indices], name="dU"), row=2, col=1)
    fig.add_vline(x=time_s[center], line_color="#777777", line_dash="dot",
                  annotation_text="detector center", row=1, col=1)
    for key, color in (("start_idx", "#377eb8"), ("end_idx", "#4daf4a")):
        if item.get(key) is not None:
            fig.add_vline(x=time_s[item[key]], line_color=color, line_dash="dash",
                          annotation_text=key, row=1, col=1)
    if item.get("fallback_peak_idx") is not None:
        peak_at = item["fallback_peak_idx"]
        fig.add_trace(go.Scatter(x=[time_s[peak_at]], y=[signal["edge_signal"][peak_at]],
                                 mode="markers", name="fallback signal peak",
                                 marker=dict(color="gold", symbol="star", size=13)),
                      row=1, col=1, secondary_y=True)
    if item.get("derivative_peak_idx") is not None:
        peak = item["derivative_peak_idx"]
        fig.add_trace(go.Scatter(x=[time_s[peak]], y=[derivative[peak]], mode="markers",
                                 name="derivative peak", marker=dict(color="red", size=11)), row=2, col=1)
    for side, color in (("left", "#d95f02"), ("right", "#1b9e77")):
        threshold = item.get(f"threshold_{side}", item.get("core_threshold"))
        if threshold is not None:
            for sign in (-1, 1):
                fig.add_hline(y=sign * threshold, line_dash="dot",
                              line_color=color, row=2, col=1,
                              annotation_text=f"{sign:+d} q_{side}")
    if item.get("derivative_fwhm_left_idx") is not None:
        fig.add_vrect(x0=time_s[item["derivative_fwhm_left_idx"]],
                      x1=time_s[item["derivative_fwhm_right_idx"]],
                      fillcolor="orange", opacity=0.15, line_width=0, row=2, col=1)
    fig.update_xaxes(title_text="Time (s)", row=2, col=1)
    fig.update_yaxes(title_text="clean", row=1, col=1, secondary_y=False)
    fig.update_yaxes(title_text="clean − Bcontext", row=1, col=1, secondary_y=True)
    fig.update_layout(title=f"Udeco{channel} {item['roi_id']} candidate {candidate_index}: "
                            f"{item['detector_name']}: {'resolved' if item['resolved'] else item['reason']}; start={item.get('start_rule')}; end={item.get('end_rule')}; mode={item.get('right_mode')}; sharp={item.get('is_sharp')}; left/right quiet={item.get('left_quiet_points')}/{item.get('right_quiet_points')}; limit={item.get('search_limit')}",
                      template="plotly_white", height=700, dragmode="pan",
                      hovermode="x unified")
    fig.show(config={"scrollZoom": True, "displaylogo": False})
    return fig

candidate_options = [(f"Udeco{channel} {item['roi_id']} #{index} t={time_s[item['center_idx']]:.2f}s",
                      (channel, index))
                     for channel in (2, 3) for index, item in enumerate(localized[channel])]
if candidate_options:
    first_channel, first_index = candidate_options[0][1]
    first_candidate_figure = plot_candidate_local(first_channel, first_index)
    candidate_selector = widgets.Dropdown(options=candidate_options, description="Candidate:",
                                          layout=widgets.Layout(width="750px"))
    candidate_output = widgets.Output()
    def _show_selected_candidate(change):
        with candidate_output:
            clear_output(wait=True)
            channel, index = change["new"]
            plot_candidate_local(channel, index)
    candidate_selector.observe(_show_selected_candidate, names="value")
    display(candidate_selector, candidate_output)

## 7. 每个 ROI 的差分上升/下降极值与完整边沿

正负极值一经检出就显示。成功定位者显示 `start→end`；失败者保留中心和原因。`delta` 使用 `clean−Bcontext` 的起终点值。下方的 Udeco2、Udeco3 双 y 轴图同时显示原信号、逐点差分、固定基线导数门槛和各边沿的实际阈值带。重叠时绿色菱形标出两极值之间 `|dU|` 最小的分割点。

金色星形为回退信号峰。原极值与回退边沿使用不同标记，表格保留两者；回退成对成功时主标记切换到回退结果。

橙色 `q_left` 与绿色 `q_right` 是每条候选实际采用的阈值；候选详情同时显示右侧模式、连续低差分点数及 `start_rule/end_rule`。ROI 缺少通过幅度保护的一升一降，或两主边沿幅度差超过较大幅度的 50% 时，仅取消该 ROI 的模态与事件标签。

In [ ]:
# Local override：直接展示与形态分类
min_amplitude_sigma_k = MIN_AMPLITUDE_SIGMA_K
delta_mismatch_fraction = DELTA_MISMATCH_FRACTION
deriv_peak_width_threshold = DERIV_PEAK_WIDTH_THRESHOLD
deriv_peak_sigma_k = DERIV_PEAK_SIGMA_K
direction_map = DIRECTION_MAP
roi_fill_colors = ROI_FILL_COLORS
left_lead_color = ROI_LEFT_LEAD_COLOR

classify_config = replace(localize_config, min_amplitude_sigma_k=min_amplitude_sigma_k,
                          delta_mismatch_fraction=delta_mismatch_fraction,
                          deriv_peak_width_threshold_sec=deriv_peak_width_threshold,
                          deriv_peak_sigma_k=deriv_peak_sigma_k)
tables = {channel: build_direct_event_table(localized[channel], roi_results[channel]["rois"],
                                            channel=f"Udeco{channel}",
                                            sigma_udeco=noise[channel]["sigma_udeco"], fs=fs,
                                            config=classify_config,
                                            direction_map=direction_map[channel])
          for channel in (2, 3)}
event_table = pd.concat(tables.values(), ignore_index=True)

def add_edge_segments(fig, rows, edge, name, color, *, dual_axis=False):
    segment_x, segment_y = [], []
    for _, item in rows.loc[rows.resolved == True].iterrows():
        segment = np.arange(int(item.start_idx), int(item.end_idx) + 1)
        segment_x.extend(time_s[segment])
        segment_y.extend(edge[segment])
        segment_x.append(None)
        segment_y.append(None)
    trace = go.Scattergl(x=segment_x, y=segment_y, mode="lines",
                         name=f"{name} start→end", line=dict(color=color, width=5))
    fig.add_trace(trace, secondary_y=False) if dual_axis else fig.add_trace(trace)

def plot_full_roi_edges(channel, *, dual_axis=False):
    table = tables[channel]
    edge = processed[f"Udeco{channel}"]["edge_signal"]
    derivative = np.diff(edge, prepend=edge[0])
    fig = make_subplots(specs=[[{"secondary_y": True}]]) if dual_axis else go.Figure()
    def add_primary(trace):
        fig.add_trace(trace, secondary_y=False) if dual_axis else fig.add_trace(trace)

    add_primary(go.Scattergl(x=time_s, y=edge, name="full Udeco edge_signal",
                             line=dict(color="#555", width=1)))
    if dual_axis:
        fig.add_trace(go.Scattergl(x=time_s, y=derivative, name="Udeco dU",
                                   line=dict(color="#9467bd", width=1)), secondary_y=True)
    if dual_axis:
        baseline_limit = localize_config.threshold_du * noise[channel]["sigma_du"]
        for sign in (-1, 1):
            fig.add_trace(go.Scatter(x=[time_s[0], time_s[-1]],
                                     y=[sign * baseline_limit] * 2, mode="lines",
                                     name=f"{sign:+d} baseline core floor ({localize_config.threshold_du:g} sigma)",
                                     line=dict(color="#999", dash="dot")), secondary_y=True)
        for side, color in (("left", "#d95f02"), ("right", "#1b9e77")):
            for sign in (-1, 1):
                xs, ys = [], []
                for _, item in table.iterrows():
                    threshold = item.get(f"threshold_{side}")
                    if pd.isna(threshold):
                        threshold = item.get("core_threshold")
                    if pd.isna(threshold):
                        continue
                    middle = int(item.derivative_peak_idx) if pd.notna(item.get("derivative_peak_idx")) else int(item.center_idx)
                    left = int(item.start_idx) if pd.notna(item.get("start_idx")) else middle
                    right = int(item.end_idx) if pd.notna(item.get("end_idx")) else middle
                    lo, hi = (left, middle) if side == "left" else (middle, right)
                    if lo > hi:
                        continue
                    xs.extend([time_s[lo], time_s[hi], None])
                    ys.extend([sign * threshold, sign * threshold, None])
                fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", name=f"{sign:+d} q_{side}",
                                         line=dict(color=color, dash="dot" if sign < 0 else "solid")),
                              secondary_y=True)
    fallback_rows = table.loc[table.detector_name == "peak_fallback"]
    if not fallback_rows.empty:
        peaks = fallback_rows.fallback_peak_idx.dropna().astype(int).unique()
        add_primary(go.Scattergl(x=time_s[peaks], y=edge[peaks], mode="markers",
                                 name="fallback signal peaks", marker=dict(color="gold", symbol="star", size=12)))
    for direction, name, color, symbol in ((1, "rising", "#e41a1c", "triangle-up"),
                                            (-1, "falling", "#377eb8", "triangle-down")):
        rows = table.loc[table.direction == direction]
        add_edge_segments(fig, rows, edge, name, color, dual_axis=dual_axis)
        centers = rows.center_idx.to_numpy(dtype=int)
        descriptions = [f"{item.roi_id}: {item.detector_name}; {item.classification_status}; "
                        f"reason={item.reason}; start={item.get('start_rule')}; end={item.get('end_rule')}; mode={item.get('right_mode')}; sharp={item.get('is_sharp')}; quiet={item.get('left_quiet_points')}/{item.get('right_quiet_points')}; endpoint={item.get('endpoint_source', 'turn')}; main={item.is_main_rising or item.is_main_falling}"
                        for _, item in rows.iterrows()]
        add_primary(go.Scattergl(x=time_s[centers], y=edge[centers], mode="markers",
                                 name=f"detected {name}", text=descriptions,
                                 marker=dict(color=color, size=11, symbol=["star" if d == "peak_fallback" else symbol for d in rows.detector_name]),
                                 hovertemplate="%{text}<br>t=%{x:.3f}s<extra></extra>"))
        if dual_axis:
            fig.add_trace(go.Scattergl(x=time_s[centers], y=derivative[centers],
                                       mode="markers", name=f"dU {name} extrema",
                                       marker=dict(color=color, size=8, symbol=symbol)),
                          secondary_y=True)
    for mask, name, color, symbol in ((table.resolved == False, "localization failed", "#222", "x"),
                                      (table.direction_delta_conflict, "direction/delta conflict", "#ff7f00", "x")):
        flagged = table.loc[mask]
        centers = flagged.center_idx.to_numpy(dtype=int)
        add_primary(go.Scattergl(x=time_s[centers], y=edge[centers], mode="markers",
                                 name=name, text=flagged.roi_id,
                                 marker=dict(color=color, size=14, symbol=symbol)))
    if "overlap_split_idx" in table:
        splits = table.overlap_split_idx.dropna().astype(int).unique()
        add_primary(go.Scattergl(x=time_s[splits], y=edge[splits], mode="markers",
                                 name="overlap split: min |dU|", marker=dict(symbol="diamond", color="#009E73", size=9)))
        if dual_axis:
            fig.add_trace(go.Scattergl(x=time_s[splits], y=derivative[splits], mode="markers",
                                       name="dU overlap split", marker=dict(symbol="diamond", color="#009E73", size=9)),
                          secondary_y=True)
    shapes = []
    for i, roi in enumerate(roi_results[channel]["rois"]):
        shapes.append(dict(type="rect", xref="x", yref="y domain",
                           x0=time_s[roi["left_idx"]], x1=time_s[roi["right_idx"]],
                           y0=0, y1=1, fillcolor=roi_fill_colors[i % 2],
                           opacity=0.15, line_width=0))
        if roi["search_left_idx"] < roi["left_idx"]:
            shapes.append(dict(type="rect", xref="x", yref="y domain",
                               x0=time_s[roi["search_left_idx"]], x1=time_s[roi["left_idx"]],
                               y0=0, y1=1, fillcolor=left_lead_color,
                               opacity=0.27, line_width=0))
    fig.update_layout(shapes=shapes,
                      title=f"Udeco{channel}: derivative_extrema candidates and localized edges"
                            + (" / dU" if dual_axis else ""),
                      xaxis_title="Time (s)", yaxis_title="clean - Bcontext",
                      template="plotly_white", height=550, dragmode="pan")
    if dual_axis:
        fig.update_yaxes(title_text="dU per sample", secondary_y=True)
    fig.show(config={"scrollZoom": True, "displaylogo": False})
    return fig

full_edge_figures = {}
dual_edge_figures = {}
for channel in (2, 3):
    table = tables[channel]
    blocked = (table.loc[table.classification_status.str.startswith('roi_')]
               .groupby('classification_status').roi_id.nunique().to_dict())
    print(f"Udeco{channel}: {len(table)} detected; {int(table.resolved.sum())} localized; "
          f"{int((table.classification_status == 'classified').sum())} classified; "
          f"{int(table.direction_delta_conflict.sum())} sign conflicts; blocked ROIs={blocked}")
    full_edge_figures[channel] = plot_full_roi_edges(channel)
    dual_edge_figures[channel] = plot_full_roi_edges(channel, dual_axis=True)

def plot_roi_selection(channel, roi_id):
    roi_index, roi = next((i, row) for i, row in enumerate(roi_results[channel]["rois"])
                          if row["roi_id"] == roi_id)
    edge = processed[f"Udeco{channel}"]["edge_signal"]
    rows = tables[channel].loc[tables[channel].roi_id == roi_id]
    endpoint_indices = [int(item[key]) for _, item in rows.iterrows()
                        for key in ("start_idx", "end_idx")
                        if pd.notna(item.get(key))]
    lo = min([roi["search_left_idx"], *endpoint_indices])
    hi = max([roi["right_idx"], *endpoint_indices])
    indices = np.arange(lo, hi + 1)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=time_s[indices], y=edge[indices],
                             name="Udeco edge_signal"))
    fig.add_vrect(x0=time_s[roi["left_idx"]], x1=time_s[roi["right_idx"]],
                  fillcolor=roi_fill_colors[roi_index % 2], opacity=0.20, line_width=0)
    if roi["search_left_idx"] < roi["left_idx"]:
        fig.add_vrect(x0=time_s[roi["search_left_idx"]], x1=time_s[roi["left_idx"]],
                      fillcolor=left_lead_color, opacity=0.35, line_width=0)
    fig.add_vline(x=time_s[roi["main_peak_idx"]], line_color="#666666",
                  line_dash="dot", annotation_text="Uh main peak")
    fallback_peaks = rows.loc[rows.detector_name == "peak_fallback", "fallback_peak_idx"].dropna().astype(int).unique() if "fallback_peak_idx" in rows else []
    for peak_at in fallback_peaks:
        fig.add_trace(go.Scatter(x=[time_s[peak_at]], y=[edge[peak_at]], mode="markers",
                                 name="fallback signal peak", marker=dict(color="gold", symbol="star", size=13)))
    for _, item in rows.iterrows():
        center = int(item.center_idx)
        color = "#e41a1c" if item.direction > 0 else "#377eb8"
        if item.resolved:
            segment = np.arange(int(item.start_idx), int(item.end_idx) + 1)
            fig.add_trace(go.Scatter(x=time_s[segment], y=edge[segment], mode="lines",
                                     name=f"{item.edge_role} start→end",
                                     line=dict(color=color, width=6)))
            for key, marker_color in (("start_idx", "#377eb8"),
                                      ("end_idx", "#4daf4a")):
                fig.add_vline(x=time_s[int(item[key])], line_dash="dash",
                              line_color=marker_color,
                              annotation_text=f"{item.edge_role} {key}")
        symbol = "triangle-up" if item.direction > 0 else "triangle-down"
        label = (f"{item.detector_name} {item.edge_role}: {item.classification_status}; "
                 f"reason={item.reason}; start={item.get('start_rule')}; end={item.get('end_rule')}; mode={item.get('right_mode')}; sharp={item.get('is_sharp')}; quiet={item.get('left_quiet_points')}/{item.get('right_quiet_points')}; endpoint={item.get('endpoint_source', 'turn')}; main={item.is_main_rising or item.is_main_falling}")
        fig.add_trace(go.Scatter(x=[time_s[center]], y=[edge[center]], mode="markers",
                                 name=label, marker=dict(size=14, color=color, symbol="star" if item.detector_name == "peak_fallback" else symbol),
                                 hovertemplate="%{fullData.name}<br>t=%{x:.3f}s<extra></extra>"))
        if not item.resolved or item.direction_delta_conflict:
            fig.add_trace(go.Scatter(x=[time_s[center]], y=[edge[center]], mode="markers",
                                     name="localization failed" if not item.resolved else "sign conflict",
                                     marker=dict(size=17, color="#222" if not item.resolved
                                                 else "#ff7f00", symbol="x")))
    status = rows.pair_status.iloc[0] if not rows.empty else "no_candidate"
    fig.update_layout(title=f"Udeco{channel} {roi_id}: {status}; derivative_extrema candidates",
                      xaxis_title="Time (s)", yaxis_title="clean - Bcontext",
                      template="plotly_white", height=550, dragmode="pan")
    fig.show(config={"scrollZoom": True, "displaylogo": False})
    return fig

roi_options = [(f"Udeco{channel} {roi['roi_id']}", (channel, roi['roi_id']))
               for channel in (2, 3) for roi in roi_results[channel]["rois"]]
if roi_options:
    first_roi_figure = plot_roi_selection(*roi_options[0][1])
    roi_selector = widgets.Dropdown(options=roi_options, description="ROI:",
                                    layout=widgets.Layout(width="500px"))
    roi_output = widgets.Output()
    def _show_selected_roi(change):
        with roi_output:
            clear_output(wait=True)
            plot_roi_selection(*change["new"])
    roi_selector.observe(_show_selected_roi, names="value")
    display(roi_selector, roi_output)

## 8. Derivative morphology、分类与 event_table

两条主边沿均定位成功、分别通过严格的 2σ_udeco 幅度门槛且幅度相差不超过 50% 时，正向主边沿的尖锐判据决定 ROI 为 detach-attach 或 release-force。负向边沿保留自身形态诊断；导数峰强度、峰宽和 T_edge 只用于观察。未通过配对检查的 ROI 取消事件标签，完整表格保留失败原因。


In [ ]:
# Local override：散点筛选与表格显示列
plot_classified_only = True
table_columns = ["channel", "roi_id", "center_idx", "direction", "detector_name",
                 "detector_score", "resolved", "reason", "endpoint_source", "search_limit", "fallback_peak_idx", "delta", "baseline_snr",
                 "amplitude_pass", "classification_status", "direction_delta_conflict",
                 "start_idx", "end_idx", "start_rule", "end_rule", "right_mode", "is_sharp", "left_quiet_points", "right_quiet_points", "threshold_left", "threshold_right", "T_edge",
                 "derivative_peak", "normalized_derivative_peak",
                 "derivative_peak_width", "monotonicity", "derivative_support",
                 "is_main_rising", "is_main_falling", "main_source", "fallback_used", "edge_role", "pair_status",
                 "overlap_duplicate", "duplicate_owner_roi_id", "morphology", "roi_mode", "event_label"]

plot_table = event_table[event_table.event_label.notna()] if plot_classified_only else event_table[event_table.resolved]
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("T_edge vs normalized derivative peak",
                                    "T_edge vs derivative peak FWHM"))
for channel in (2, 3):
    channel_table = plot_table[plot_table.channel == f"Udeco{channel}"]
    for morphology, group in channel_table.groupby("morphology"):
        color = "#e41a1c" if morphology.startswith("sharp") else "#377eb8"
        fig.add_trace(go.Scatter(x=group.T_edge, y=group.normalized_derivative_peak,
                                 mode="markers", name=f"Udeco{channel} {morphology}",
                                 marker=dict(color=color, symbol="circle" if channel == 2 else "diamond"),
                                 text=group.roi_id, hovertemplate="%{text}<br>T_edge=%{x:.3f}s<br>peak/σdu=%{y:.1f}<extra></extra>"), row=1, col=1)
        fig.add_trace(go.Scatter(x=group.T_edge, y=group.derivative_peak_width,
                                 mode="markers", name=f"Udeco{channel} {morphology}",
                                 marker=dict(color=color, symbol="circle" if channel == 2 else "diamond"),
                                 showlegend=False, text=group.roi_id), row=1, col=2)
for col in (1, 2):
    fig.update_xaxes(title_text="T_edge (s)", row=1, col=col)
fig.add_hline(y=deriv_peak_width_threshold, line_dash="dot", line_color="gray", row=1, col=2)
fig.update_yaxes(title_text="abs(derivative peak) / baseline sigma_du", row=1, col=1)
fig.update_yaxes(title_text="Derivative FWHM (s)", row=1, col=2)
fig.update_layout(title="Classified derivative extrema morphology", template="plotly_white",
                  height=500, width=1150, dragmode="pan")
fig.show(config={"scrollZoom": True, "displaylogo": False})
print("Classified morphology counts:")
display(plot_table.groupby(["channel", "morphology", "event_label"]).size().rename("count").reset_index())
print("Full event_table (Pandas may truncate display; variable retains every row):")
display(event_table.reindex(columns=table_columns))

## 9. 四路时间图：ROI 模态与分割线

Uh2、Uh3、Udeco2、Udeco3 分别使用单 y 轴显示 `clean−Bcontext`。背景色表示该 ROI 最终为 detach-attach、release-force 或取消分类；竖线为逻辑 ROI 边界。最终图的图例只说明三类底色。


In [ ]:
# Local override：最终 ROI 模态时间图
final_roi_colors = {"detach-attach": "#e8a07e", "release-force": "#86bbd8",
                    "cancelled": "#b8b8b8"}
final_roi_opacity = 0.20
final_boundary_color = "#c9c9c9"


def plot_final_roi_modes(channel, sensor):
    name = f"{sensor}{channel}"
    signal = processed[name]["edge_signal"]
    fig = go.Figure()
    fig.add_trace(go.Scattergl(x=time_s, y=signal, mode="lines",
                               line=dict(color="#33415c", width=1.2),
                               hovertemplate="t=%{x:.3f}s<br>edge_signal=%{y:.5g}<extra></extra>",
                               showlegend=False))
    shapes = []
    for roi in roi_results[channel]["rois"]:
        roi_rows = tables[channel].loc[tables[channel].roi_id == roi["roi_id"]]
        mode = roi_rows.roi_mode.iloc[0] if not roi_rows.empty else "cancelled"
        left, right = time_s[roi["left_idx"]], time_s[roi["right_idx"]]
        shapes.append(dict(type="rect", xref="x", yref="paper",
                           x0=left, x1=right, y0=0, y1=1,
                           fillcolor=final_roi_colors[mode], opacity=final_roi_opacity,
                           line=dict(width=0)))
        for boundary in (left, right):
            shapes.append(dict(type="line", xref="x", yref="paper",
                               x0=boundary, x1=boundary, y0=0, y1=1, opacity=0.6,
                               line=dict(color=final_boundary_color, width=0.8, dash="dash")))
    fig.update_layout(shapes=shapes)
    for mode, color in final_roi_colors.items():
        fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name=mode,
                                 marker=dict(symbol="square", size=13, color=color),
                                 showlegend=True))
    fig.update_layout(title=f"{name}: ROI modality", xaxis_title="Time (s)",
                      yaxis_title="clean − Bcontext", template="plotly_white",
                      height=480, dragmode="pan", legend_title_text="ROI background")
    fig.show(config={"scrollZoom": True, "displaylogo": False})
    return fig


final_roi_figures = {}
for sensor in ("Uh", "Udeco"):
    for channel in (2, 3):
        final_roi_figures[(sensor, channel)] = plot_final_roi_modes(channel, sensor)
